# Урок 17. Recommender Systems — рекомендательные системы
**Библиотеки:** numpy, pandas

**Цель:** понять user-item матрицу и построить рекомендации «похожих пользователей».

## Простыми словами
Netflix, Spotify, маркетплейсы — всё это рекомендательные системы. Два главных подхода:
- **Content-based:** «тебе нравились боевики → вот ещё боевик». Сравниваем ПРИЗНАКИ товаров.
- **Collaborative filtering:** «людям, похожим на тебя, понравилось X → попробуй X».
  Признаки товара не нужны вообще — только оценки!

**User-item матрица:** строки — люди, столбцы — фильмы, в ячейках оценки. Много пропусков —
никто не смотрел всё. Задача = предсказать пропуски.

**Похожесть** меряем косинусной близостью векторов оценок: 1 = одинаковые вкусы, 0 = ничего общего.

In [ ]:
import numpy as np
import pandas as pd

ratings = pd.DataFrame({
    'Матрица':      [5, 4, np.nan, 1, np.nan],
    'Титаник':      [1, np.nan, 5, 5, 2],
    'Терминатор':   [5, 5, 1, np.nan, 4],
    'Дневник памяти':[np.nan, 1, 5, 4, 1],
    'Интерстеллар': [4, 5, np.nan, 2, np.nan],
}, index=['Аня', 'Борис', 'Вера', 'Галя', 'Дима'])
print(ratings)

def cosine_sim(a, b):
    # сравниваем только фильмы, которые оценили ОБА
    mask = a.notna() & b.notna()
    if mask.sum() == 0: return 0.0
    va, vb = a[mask].values, b[mask].values
    return (va @ vb) / (np.linalg.norm(va) * np.linalg.norm(vb))

target = 'Дима'
sims = {u: cosine_sim(ratings.loc[target], ratings.loc[u])
        for u in ratings.index if u != target}
sims = pd.Series(sims).sort_values(ascending=False)
print('\nПохожесть на Диму:'); print(sims.round(3))

In [ ]:
# Рекомендация: что любит самый похожий пользователь из того, что Дима не смотрел
best_match = sims.index[0]
unseen = ratings.loc[target][ratings.loc[target].isna()].index
candidates = ratings.loc[best_match, unseen].dropna().sort_values(ascending=False)
print(f'Самый похожий на Диму: {best_match}')
print('Рекомендации Диме:')
print(candidates)

## Что произошло
Мы посчитали похожесть Димы с каждым по ПЕРЕСЕЧЕНИЮ оценённых фильмов. Самым похожим оказался
любитель фантастики — и Диме рекомендуются его любимые фильмы, которые Дима ещё не видел.
Это user-based collaborative filtering в миниатюре. В реальном Netflix то же самое,
только матрица миллионы × десятки тысяч, и используется матричная факторизация.

## Типичные ошибки
1. Считать похожесть по всем фильмам с NaN — получится мусор. Только по общим оценкам.
2. Cold start: новому пользователю без оценок рекомендовать нечем — нужен content-based или топ-популярное.
3. Рекомендовать уже просмотренное — всегда фильтруй уже оценённое.

## Конспект 📓
User-item матрица: люди × товары, в ячейках оценки, много NaN. Косинусная близость = похожесть вкусов.
User-based CF: найди похожих → возьми их высокие оценки на непросмотренное. Cold start — главная боль.

---
## Мини-задание 💪
Сделай рекомендацию не по одному похожему пользователю, а по топ-2: усредни их оценки непросмотренных фильмов с весом похожести.

## 5 проверочных вопросов ❓
1. Content-based vs collaborative — разница на примере?
2. Что лежит в user-item матрице и почему там много пропусков?
3. Как косинусная близость измеряет похожесть вкусов?
4. Что такое cold start?
5. Почему нельзя считать похожесть по фильмам с NaN?

## Домашнее задание 🏠
Задача 47 из practice_tasks.md. Расширь матрицу до 8 пользователей и 10 фильмов, построй рекомендации для двоих.
